# 01 – Exploration du dataset C-MAPSS (FD001)

Objectif : comprendre la structure des données, repérer les capteurs utiles et visualiser la dégradation des moteurs.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))  # pour importer src/

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data.load import load_cmapss, add_rul, constant_columns, SENSOR_COLS, SETTING_COLS

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Chargement

In [ ]:
train, test, rul_test = load_cmapss("FD001")
print("train :", train.shape, "| moteurs :", train.unit.nunique())
print("test  :", test.shape, "| moteurs :", test.unit.nunique())
print("RUL test :", rul_test.shape)
train.head()

In [ ]:
train.describe().T

## 2. Durée de vie des moteurs

Combien de cycles chaque moteur tient-il avant la panne ?

In [ ]:
lifetimes = train.groupby("unit")["cycle"].max()
print(lifetimes.describe())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lifetimes, bins=20, color="steelblue", edgecolor="white")
ax.set(xlabel="Durée de vie (cycles)", ylabel="Nombre de moteurs", title="Distribution des durées de vie – FD001 train")
plt.show()

## 3. Capteurs constants

Un capteur qui ne varie pas n'apporte aucune information : on le supprimera.

In [ ]:
const = constant_columns(train)
print("Colonnes constantes :", const)

std = train[SETTING_COLS + SENSOR_COLS].std().sort_values()
std.head(10)

## 4. Évolution des capteurs jusqu'à la panne

On aligne les moteurs sur la panne (axe x = RUL brute, 0 = panne) pour voir les tendances de dégradation.

In [ ]:
df = add_rul(train, clip=None)
useful = [c for c in SENSOR_COLS if c not in const]
units = df.unit.unique()[:10]  # 10 moteurs

fig, axes = plt.subplots(len(useful) // 3 + 1, 3, figsize=(15, 3 * (len(useful) // 3 + 1)))
for ax, s in zip(axes.flat, useful):
    for u in units:
        d = df[df.unit == u]
        ax.plot(d["RUL"], d[s], lw=0.8, alpha=0.7)
    ax.set_title(s)
    ax.invert_xaxis()  # la panne à droite
    ax.set_xlabel("RUL (cycles)")
for ax in axes.flat[len(useful):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Corrélation des capteurs avec la RUL

In [ ]:
corr = df[useful + ["RUL"]].corr()["RUL"].drop("RUL").sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
corr.plot.barh(ax=ax, color=["indianred" if v < 0 else "steelblue" for v in corr])
ax.set(title="Corrélation de chaque capteur avec la RUL", xlabel="Corrélation de Pearson")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[useful].corr(), cmap="RdBu_r", center=0, annot=True, fmt=".1f", ax=ax)
ax.set_title("Corrélations entre capteurs")
plt.show()

## 6. Le jeu de test

Les trajectoires de test s'arrêtent avant la panne. La vraie RUL au dernier cycle est dans `RUL_FD001.txt`.

In [ ]:
test_len = test.groupby("unit")["cycle"].max()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(test_len, bins=20, color="seagreen", edgecolor="white")
axes[0].set(title="Longueur des trajectoires de test", xlabel="Cycles observés")
axes[1].hist(rul_test, bins=20, color="darkorange", edgecolor="white")
axes[1].set(title="Vraie RUL des moteurs de test", xlabel="RUL (cycles)")
plt.tight_layout()
plt.show()

## 7. Conclusions

_À compléter après exécution :_
- Capteurs à supprimer :
- Capteurs les plus informatifs :
- Capteurs très corrélés entre eux (redondants) :
- Durée de vie moyenne / min / max :